In [1]:
import pandas as pd
games_df = pd.read_csv(r'F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\games.csv')
games_df

,game_id,game_key,season,season_type,week,home_team_abbr,visitor_team_abbr
0,2023090700,59173,2023,REG,1,KC,DET
1,2023091000,59174,2023,REG,1,ATL,CAR
2,2023091001,59175,2023,REG,1,BAL,HOU
3,2023091002,59176,2023,REG,1,CLE,CIN
4,2023091003,59177,2023,REG,1,IND,JAX
...,...,...,...,...,...,...,...
997,2026011800,60172,2025,POST,20,CHI,LAR
998,2026011801,60173,2025,POST,20,NE,HOU
999,2026012500,60174,2025,POST,21,DEN,NE
1000,2026012501,60175,2025,POST,21,SEA,LAR


In [4]:
games_df.columns.tolist()

['game_id',
 'game_key',
 'season',
 'season_type',
 'week',
 'home_team_abbr',
 'visitor_team_abbr']

In [5]:

games_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   game_id            1002 non-null   int64
 1   game_key           1002 non-null   int64
 2   season             1002 non-null   int64
 3   season_type        1002 non-null   str  
 4   week               1002 non-null   int64
 5   home_team_abbr     1002 non-null   str  
 6   visitor_team_abbr  1002 non-null   str  
dtypes: int64(4), str(3)
memory usage: 183.7 KB


In [6]:
missing_summary = pd.DataFrame({
    "missing_count": games_df.isna().sum(),
    "missing_pct": (games_df.isna().mean() * 100).round(2)
})

display(missing_summary)

,missing_count,missing_pct
game_id,0,0.0
game_key,0,0.0
season,0,0.0
season_type,0,0.0
week,0,0.0
home_team_abbr,0,0.0
visitor_team_abbr,0,0.0


In [7]:
print("Duplicate Rows:", games_df.duplicated().sum())

print("Duplicate game_id:",
      games_df["game_id"].duplicated().sum())

print("Missing game_id:",
      games_df["game_id"].isna().sum())

print("Duplicate game_key:",
      games_df["game_key"].duplicated().sum())

Duplicate Rows: 0
Duplicate game_id: 0
Missing game_id: 0
Duplicate game_key: 0


In [8]:
for col in ["season", "season_type", "week"]:
    print(f"\n{col}")
    display(games_df[col].value_counts(dropna=False).sort_index())

print("Same Home and Visitor Team:",
      (games_df["home_team_abbr"] ==
       games_df["visitor_team_abbr"]).sum())

print("Unique Home Teams:",
      games_df["home_team_abbr"].nunique())

print("Unique Visitor Teams:",
      games_df["visitor_team_abbr"].nunique())


season


season
2023    334
2024    334
2025    334
Name: count, dtype: int64


season_type


season_type
POST     39
PRE     147
REG     816
Name: count, dtype: int64


week


week
0      3
1     96
2     96
3     96
4     48
5     42
6     44
7     43
8     45
9     43
10    42
11    43
12    43
13    45
14    42
15    48
16    48
17    48
18    48
19    18
20    12
21     6
23     3
Name: count, dtype: int64

Same Home and Visitor Team: 0
Unique Home Teams: 33
Unique Visitor Teams: 33


In [11]:
games_ids = set(games_df["game_id"])
player_play_game_ids = set(games_df["game_id"])

print("Games without player_play records:",
      len(games_ids - player_play_game_ids))

print("player_play games missing from games table:",
      len(player_play_game_ids - games_ids))

Games without player_play records: 0
player_play games missing from games table: 0


# Games Table — Data Profiling

**Dataset:** `games.csv`

**Objective:** Assess the structure, completeness, uniqueness, and referential integrity of the games dataset.

**Expected Granularity:** One record per NFL game.

**Profiling Checks:**
1. Dataset structure and data types
2. Missing values
3. Duplicate records and primary key validation
4. Season, week, and team consistency
5. Referential integrity with `player_play.csv`

**Expected Primary Key:** `game_id`

**Status:** Profiling in progress.

In [12]:
all_teams = sorted(
    set(games_df["home_team_abbr"].dropna()) |
    set(games_df["visitor_team_abbr"].dropna())
)

print(all_teams)
print("Total unique abbreviations:", len(all_teams))

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LAR', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']
Total unique abbreviations: 33


### Team Abbreviation Consistency

**Finding:**
- 33 unique team abbreviations were identified.
- Both `LA` and `LAR` appear in the dataset and refer to the Los Angeles Rams.
- Therefore, the 33 abbreviations represent 32 distinct NFL teams.

**Data Quality Issue:** Inconsistent categorical representation of the same team.

**Preprocessing Recommendation:**
- Standardize `LA` and `LAR` to a consistent abbreviation, preferably `LAR`.
- Ensure consistent team naming across related datasets before joining or aggregating data.
- Preserve original values during data profiling.

In [13]:
player_play_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\player_play.csv")  

print("Duplicate game_id:",
      games_df["game_id"].duplicated().sum())

print("Missing game_id:",
      games_df["game_id"].isna().sum())

print("Duplicate game_key:",
      games_df["game_key"].duplicated().sum())

Duplicate game_id: 0
Missing game_id: 0
Duplicate game_key: 0


In [14]:
games_ids = set(games_df["game_id"])
player_play_ids = set(player_play_df["game_id"])

print("Games without player_play records:",
      len(games_ids - player_play_ids))

print("Player_play games missing from games:",
      len(player_play_ids - games_ids))

Games without player_play records: 1
Player_play games missing from games: 0


In [15]:
unmatched_games = games_df[
    games_df["game_id"].isin(
        games_ids - player_play_ids
    )
]

display(unmatched_games)

,game_id,game_key,season,season_type,week,home_team_abbr,visitor_team_abbr
57,2023100108,59230,2023,REG,4,PHI,WAS


## Final Data Profiling Summary — games.csv

### Dataset Overview
- Total records: 1,002
- Total columns: 7
- Seasons covered: 2023, 2024, 2025
- Games per season: 334
- Expected granularity: One record per game

### Season Distribution
- Regular Season (REG): 816 games
- Preseason (PRE): 147 games
- Postseason (POST): 39 games

### Data Validation Findings
- No games have identical home and visitor team abbreviations.
- 33 unique team abbreviations were identified, including `LA` and `LAR`, which both represent the Los Angeles Rams.
- Week values range from 0 to 23, with Week 22 not represented.

### Referential Integrity Findings
- `games.csv` contains 1,002 games.
- `player_play.csv` contains 1,001 unique games.
- One game in `games.csv` has no corresponding records in `player_play.csv`:
  - Game ID: `2023100108`
  - Season: 2023
  - Season Type: REG
  - Week: 4
  - Home Team: PHI
  - Visitor Team: WAS

### Data Quality Decisions
- Preserve the unmatched game in the original dataset.
- Document the cross-table coverage gap.
- Standardize team abbreviations during preprocessing, after checking conventions across related datasets.
- Do not automatically remove unmatched games.

### Conclusion
The games dataset has generally plausible season, week, and team distributions. One cross-table coverage gap was identified between `games.csv` and `player_play.csv`.

**Status: Initial Data Profiling Completed, subject to primary key and missing-value validation results.**